## Baseline Character Tokenizer
#### Character-level tokenizer using English, Turkish, and Chinese training data evaluating on validation data and saving vocabulary for language-model experimentation

### 1. Environment & Imports

In [1]:
# imports
import sys # for Python information
import torch # Pytorch for machine learning tasks
from pathlib import Path # filepath tool
import json # for JSON data storage

print("Python:", sys.version.split()[0]) # print version of Python
print("PyTorch:", torch.__version__) # print version of Pytorch
print("CUDA available:", torch.cuda.is_available()) # check Pytorch access to CUDA (GPU computation)

Python: 3.12.16
PyTorch: 2.4.1+cu121
CUDA available: True


### 2. Load Multilingual Training Data

In [2]:
# Load multilingual training data
DATA_DIR = Path("/srv/data/lt2326-h26/a1") # location of data

train_texts = {} # dictionary to hold 3 language collections

for lang in ["en", "tr", "zh"]:
    file_path = DATA_DIR / "train" / f"{lang}.txt" # use training data only

    with open(file_path, "r", encoding="utf-8") as f: # open file (training data)
        train_texts[lang] = f.read().splitlines() # add list of sentences to train_texts dictionary; each iteration is a different language (en, tr, zh)

    print(f"{lang}: {len(train_texts[lang]):,} sentences")

en: 36,004 sentences
tr: 30,379 sentences
zh: 75,962 sentences


### 3. Build the Character Vocabulary

In [3]:
# Identify unique characters across the training languages
unique_chars = set() # set to hold vocabulary of unique characters

for lang in ["en", "tr", "zh"]: # loop over the three linguistic datasets
    for sentence in train_texts[lang]: # loop over each sentence in the dataset
        unique_chars.update(sentence)  # add the sentence characters to the vocabulary set

chars = sorted(unique_chars) # return list of vocabulary set in ascending Unicode code-point order

print(f"Total unique characters: {len(chars):,}") # print number of unique characters
print("First 50 characters:", chars[:50]) # print sample of the first 50 characters

Total unique characters: 9,442
First 50 characters: [' ', '!', '"', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<', '>', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T']


In [4]:
# Create character-to-ID and ID-to-character mappings
UNK_TOKEN = "<UNK>" # unknown character (in case encountered during validation/testing)
UNK_ID = 0 # unknown character ID

char_to_id = {char: i + 1 for i, char in enumerate(chars)} # dictionary to assign ID numbers to characters, starting from 1
id_to_char = {i + 1: char for i, char in enumerate(chars)} # dictionary to convert IDs to characters

id_to_char[UNK_ID] = UNK_TOKEN # value for unknown token ID key in id_to_char dictionary

vocab_size = len(chars) + 1 # size of vocabulary = total characters plus unknown character

print(f"Vocabulary size (including UNK): {vocab_size:,}") # print size of vocabulary
print("Character 'a' ID:", char_to_id.get("a", UNK_ID)) # print sample character id (for 'a'); if no 'a' exists, then unknown character
print("Character '中' ID:", char_to_id.get("中", UNK_ID)) # print sample character id (for '中'); if no 'a' exists, then unknown character

Vocabulary size (including UNK): 9,443
Character 'a' ID: 58
Character '中' ID: 1022


### 4. Encode, Decode, and Test

In [5]:
# Encode and decode text using the character vocabulary
def encode(text): # function to encode text
    return [char_to_id.get(char, UNK_ID) for char in text] # return list of character IDs in text


def decode(token_ids): # function to decode text
    return "".join(id_to_char[token_id] for token_id in token_ids) # return characters from encoded text and combine into string


sample = "Hello world!" # sample text

encoded = encode(sample) # function call to encode sample text
decoded = decode(encoded) # function call to decode sample text

print("Original:", sample) # print sample text
print("Encoded:", encoded) # print encoded IDs from sample
print("Decoded:", decoded) # print decoded text from encodings

Original: Hello world!
Encoded: [38, 62, 69, 69, 72, 1, 80, 72, 75, 69, 61, 2]
Decoded: Hello world!


In [6]:
# Test character tokenizer on all three languages
for lang in ["en", "tr", "zh"]: # loop over each language collection
    sample = train_texts[lang][0] # sample first line from each language

    encoded = encode(sample) # encode sample data
    decoded = decode(encoded) # decode sample data

    print(f"\nLanguage: {lang}") # print language label 
    print("Original:", sample) # print initial sample data (first line)
    print("Number of characters:", len(sample)) # print number of characters in line (should match number of tokens below)
    print("Number of tokens:", len(encoded)) # print number of tokens in line (should match number of characters above)
    print("Decoded matches original:", decoded == sample) # return true if decoded line is the same as the original line


Language: en
Original: Since Jack Kirby had more than enough assignments, Lee assigned John Buscema the task of illustrating the new book."
Number of characters: 116
Number of tokens: 116
Decoded matches original: True

Language: tr
Original: Ekonomi alanındaki bu gelişmelerin yanında, okuma-yazma bilmezliğe karşı yoğun bir savaşım görüldü.
Number of characters: 99
Number of tokens: 99
Decoded matches original: True

Language: zh
Original: 他直到死前都沒有親自管理過公國。
Number of characters: 16
Number of tokens: 16
Decoded matches original: True


In [7]:
# Test handling of an unknown character

unknown_char = "\U0001F6F8" # sample unknown character unlikely to be in vocabulary

print("Character:", unknown_char) # print unknown character
print("In vocabulary:", unknown_char in char_to_id) # return false if character not in vocabulary

encoded = encode(unknown_char) # function call to encode sample character
decoded = decode(encoded) # function call to decode sample character

print("Encoded:", encoded) # print encoded ID for unknown character
print("Decoded:", decoded) # print decoded unknown character

Character: 🛸
In vocabulary: False
Encoded: [0]
Decoded: <UNK>


### 5. Evaluate on Validation Data

In [8]:
# Load multilingual validation data
valid_texts = {} # dictionary to hold validation texts

for lang in ["en", "tr", "zh"]: # loop over language data collections
    file_path = DATA_DIR / "valid" / f"{lang}.txt" # define filepath

    with open(file_path, "r", encoding="utf-8") as f: # open validation data in read mode
        valid_texts[lang] = f.read().splitlines() # add list of sentences to validation text dictionary; each iteration is a different language (en, tr, zh)

    print(f"{lang}: {len(valid_texts[lang]):,} validation sentences") # print number of validation sentences per language

en: 4,540 validation sentences
tr: 3,735 validation sentences
zh: 9,554 validation sentences


In [9]:
# Calculate validation statistics
char_stats = {} # dictionary to hold statistics by language

for lang in ["en", "tr", "zh"]: # loop over each language collection
    sentences = valid_texts[lang] # list of validation sentences for given language

    total_sentences = len(sentences) # total number of sentences for given language
    total_chars = sum(len(sentence) for sentence in sentences) # total number of characters per language
    total_tokens = sum(len(encode(sentence)) for sentence in sentences) # total number of tokens per language

    unknown_count = sum( # total number of unknown character tokens per language
        sum(token_id == UNK_ID for token_id in encode(sentence)) # counts unknown tokens in a given sentence
        for sentence in sentences
    )

    char_stats[lang] = { # populate statistics dictionary
        "vocab_size": vocab_size,
        "total_sentences": total_sentences,
        "total_characters": total_chars,
        "total_tokens": total_tokens,
        "tokens_per_sentence": total_tokens / total_sentences,
        "characters_per_token": total_chars / total_tokens,
        "unknown_count": unknown_count,
        "unknown_rate": unknown_count / total_tokens
    }

    print(f"\nLanguage: {lang}")
    for metric, value in char_stats[lang].items(): # loop over each statistic within language

        # if value is float, print to 4 decimal places, otherwise print value (number formatted with comma as thousandths separator)
        print(f"{metric}: {value:,.4f}" if isinstance(value, float) else f"{metric}: {value:,}") 


Language: en
vocab_size: 9,443
total_sentences: 4,540
total_characters: 368,269
total_tokens: 368,269
tokens_per_sentence: 81.1165
characters_per_token: 1.0000
unknown_count: 49
unknown_rate: 0.0001

Language: tr
vocab_size: 9,443
total_sentences: 3,735
total_characters: 368,383
total_tokens: 368,383
tokens_per_sentence: 98.6300
characters_per_token: 1.0000
unknown_count: 15
unknown_rate: 0.0000

Language: zh
vocab_size: 9,443
total_sentences: 9,554
total_characters: 368,266
total_tokens: 368,266
tokens_per_sentence: 38.5457
characters_per_token: 1.0000
unknown_count: 258
unknown_rate: 0.0007


### 6. Save and Reload the Tokenizer

In [10]:
# Save character tokenizer vocabulary and configuration
TOKENIZER_DIR = Path("../models/tokenizers") # directory path for saving JSON vocabulary file
TOKENIZER_DIR.mkdir(parents=True, exist_ok=True) # create directory if none

vocab_data = { # data for JSON vocabulary file
    "tokenizer_type": "character", # metadata
    "unk_token": UNK_TOKEN, # unknown token 
    "unk_id": UNK_ID, # unknown token ID
    "char_to_id": char_to_id # vocabulary with character to ID information
}

output_path = TOKENIZER_DIR / "character_vocab.json" # filepath for JSON vocabulary file

with open(output_path, "w", encoding="utf-8") as f: # write vocabulary file to path
    json.dump(vocab_data, f, ensure_ascii=False, indent=2) # write vocabulary file with metadata

print("Saved vocabulary to:", output_path.resolve()) # print completed save with filepath

Saved vocabulary to: /home/gussougr@GU.GU.SE/lt2326-assignment1/models/tokenizers/character_vocab.json


In [11]:
# Reload and verify saved character vocabulary
with open(output_path, "r", encoding="utf-8") as f: # open vocabulary JSON file in read mode
    loaded_vocab = json.load(f) # read vocabulary file/assign variable

print("Tokenizer type:", loaded_vocab["tokenizer_type"]) # print metadata
print("Vocabulary size:", len(loaded_vocab["char_to_id"]) + 1) # print number of vocabulary items
print("Vocabulary matches original:", loaded_vocab["char_to_id"] == char_to_id) # print check whether file vocab matches original dictionary
print("Unknown ID matches:", loaded_vocab["unk_id"] == UNK_ID) # print check whether file unknown token ID matches original

Tokenizer type: character
Vocabulary size: 9443
Vocabulary matches original: True
Unknown ID matches: True
